In [ ]:
import os
import json
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TextStreamer, BitsAndBytesConfig
from huggingface_hub import login
import gradio as gr

hf_token = os.getenv("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

In [ ]:
# LLAMA = "meta-llama/Meta-Llama-3-8B-Instruct"
LLAMA = "meta-llama/Llama-3.2-3B-Instruct"

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4"
)

In [ ]:
schemas = {
        "inventory": """
Return JSON list only.
Each row format:
{
 "item_name":"",
 "brand":"",
 "category":"",
 "color":"",
 "room":"",
 "container":"",
 "shelf":"",
 "owner":""
}
""",

        "search_queries": """
Return JSON list only.
Each row format:
{
 "query":"",
 "expected_item_name":""
}
Generate natural human search phrases.
""",

        "intent_commands": """
Return JSON list only.
Each row format:
{
 "input":"",
 "output":{
   "intent":"",
   "item_name":"",
   "room":"",
   "container":""
 }
}
""",

        "permissions": """
Return JSON list only.
Each row format:
{
 "user_id":"",
 "allowed_rooms":[]
}
""",

        "noisy_queries": """
Return JSON list only.
Each row format:
{
 "query":"",
 "intended_item":""
}
Include typo, shorthand, broken grammar.
"""
    }

In [ ]:
def generate_dataset(model, schemas, dataset_type, number_of_data, quant=True, max_new_tokens=3000):
    tokenizer = AutoTokenizer.from_pretrained(model)
    tokenizer.pad_token = tokenizer.eos_token

    if dataset_type in schemas:
        data_format = schemas[dataset_type]
    else:
        data_format = """
        Return JSON list only.
        You have to generate the most appropriate dataset in JSON format.
        Do not include "dataset_type": "" in the generated dataset.
        """

    system_prompt = f"""
        You are a professional synthetic dataset generator.

        Generate {number_of_data} rows for dataset_type = {dataset_type}

        {data_format}

        Rules:
        - realistic household data
        - no duplicates
        - diverse rooms/items
        - valid JSON only
        - no markdown
    """
    
    messages = [
        {"role":"system","content":system_prompt},
        {"role":"user","content":"Generate the dataset by returning the dataset in JSON format only. No Markdown."}
    ]

    input = tokenizer.apply_chat_template(
        messages,
        return_dict=True, # return raw Tensor instead of BatchEncoding object
        add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    input_ids = input["input_ids"]
    attention_mask = input["attention_mask"]

    streamer = TextStreamer(
        tokenizer, 
        skip_prompt=True, 
        skip_special_tokens=True
    )

    if quant:
        model = AutoModelForCausalLM.from_pretrained(model, quantization_config=quant_config).to("cuda")
    else:
        model = AutoModelForCausalLM.from_pretrained(model).to("cuda")

    outputs = model.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        streamer=streamer
    )

    # Decode only generated part
    generated_tokens = outputs[0][input_ids.shape[1]:]
    generated_text = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Save to txt file
    output_file = f'{dataset_type}_dataset.txt'
    with open(output_file, "w", encoding="utf-8") as f:
        f.write(generated_text)

    print(f"Saved output to {output_file}")
    
    

In [ ]:
generate_dataset(LLAMA, schemas, 'inventory', 3, quant=True, max_new_tokens=1000)

In [ ]:
generate_dataset(LLAMA, schemas, 'patient_record', 3, quant=True, max_new_tokens=1000)